# Notebook 09 — Independent Project

**Your new target.** Apply everything you learned on PD-L1 to a new immunooncology target assigned by your mentor.

**This notebook covers:**
1. Target research and background
2. Structure download and preparation
3. Hotspot identification in Mol*
4. BindCraft settings for your target
5. Running the design pipeline
6. Preliminary analysis

**Companion wiki pages:** [9.1](https://rucha1796.github.io/internship-bindcraft-2026/m9_01_new_target/) | [9.2](https://rucha1796.github.io/internship-bindcraft-2026/m9_02_finding_structure/) | [9.3](https://rucha1796.github.io/internship-bindcraft-2026/m9_03_hotspots/) | [9.4](https://rucha1796.github.io/internship-bindcraft-2026/m9_04_running/)

---
> Fill in the `# ===` blocks before running. Ask your mentor if unsure.

## Step 1 — Target background

Before writing any code, answer these questions (edit the markdown cell below).

### My Target: [Protein Name]

**Gene name:** [e.g., TIGIT, LAG3, HAVCR2, ...]

**Biological function:** [1-2 sentences: what does this protein do?]

**Role in cancer immunity:** [1-2 sentences: how does it suppress immune responses?]

**Natural binding partner:** [The protein it binds to form the immune checkpoint]

**Approved drugs or clinical trials:** [Any existing therapeutics targeting this protein?]

**Why interesting:** [Why is designing a new binder for this target scientifically or therapeutically interesting?]

## Step 2 — Find and download the structure

In [ ]:
import requests, os, json
import numpy as np

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ============================================================
# FILL IN YOUR TARGET INFORMATION
TARGET_PDB_ID   = "YOUR_PDB_ID"    # e.g., "7YNW" or "5X8M" — from RCSB PDB
TARGET_CHAIN    = "A"              # which chain is your binding target
TARGET_NAME     = "MyTarget"       # short name used for output files
# ============================================================

WORK_DIR = "/content/drive/MyDrive/bindcraft"
os.makedirs(WORK_DIR, exist_ok=True)

if TARGET_PDB_ID == "YOUR_PDB_ID":
    print("⚠ Please fill in TARGET_PDB_ID above before running this cell.")
else:
    print(f"Downloading {TARGET_PDB_ID} from RCSB PDB...")
    url = f"https://files.rcsb.org/download/{TARGET_PDB_ID.upper()}.pdb"
    response = requests.get(url)
    
    if response.status_code == 200:
        raw_pdb = response.text
        print(f"✓ Downloaded {TARGET_PDB_ID}")
        
        # Extract only the target chain and remove HETATM
        clean_lines = []
        for line in raw_pdb.split("\n"):
            if (line.startswith("ATOM") and len(line) > 21 
                    and line[21] == TARGET_CHAIN):
                clean_lines.append(line)
        clean_lines.append("END")
        
        clean_pdb = "\n".join(clean_lines)
        output_path = f"{WORK_DIR}/{TARGET_NAME}.pdb"
        with open(output_path, "w") as f:
            f.write(clean_pdb)
        
        atom_count = len(clean_lines) - 1
        residues = sorted(set(l[22:26].strip() for l in clean_lines if l.startswith("ATOM")))
        
        print(f"✓ Cleaned and saved: {output_path}")
        print(f"  Chain {TARGET_CHAIN}: {atom_count} ATOM records")
        print(f"  Residue count: {len(residues)}")
        print(f"  Residue number range: {residues[0]} – {residues[-1]}")
        print()
        print("Next: Open this PDB in Mol* to verify it looks right.")
        print(f"  URL: https://molstar.org/viewer/?pdb={TARGET_PDB_ID}")
    else:
        print(f"✗ Could not download {TARGET_PDB_ID} (HTTP {response.status_code})")
        print("  Check the PDB ID is correct.")

## Step 3 — Define hotspot residues

After examining the structure in Mol* and reviewing the literature, fill in your hotspot residues below.

In [ ]:
# ============================================================
# FILL IN YOUR HOTSPOT RESIDUES
HOTSPOT_RESIDUES = []   # e.g., [45, 47, 89, 92, 100]
HOTSPOT_JUSTIFICATION = {
    # residue: "reason"
    # e.g., 45: "H-bond to natural binding partner Glu32",
}
# ============================================================

if not HOTSPOT_RESIDUES:
    print("⚠ No hotspot residues defined yet.")
    print("  1. Open your target PDB in Mol*: https://molstar.org/viewer/")
    print("  2. Find the binding interface (using complex structure or literature)")
    print("  3. Identify 3-6 key residues at the interface")
    print("  4. Add them to HOTSPOT_RESIDUES = [...]")
else:
    hotspot_str = ",".join(str(r) for r in HOTSPOT_RESIDUES)
    print(f"Target: {TARGET_NAME}")
    print(f"Chain: {TARGET_CHAIN}")
    print(f"Hotspot residues: {HOTSPOT_RESIDUES}")
    print(f"Formatted for BindCraft: \"{hotspot_str}\"")
    print()
    if HOTSPOT_JUSTIFICATION:
        print("Justification:")
        for res, reason in HOTSPOT_JUSTIFICATION.items():
            print(f"  Residue {res}: {reason}")
    print()
    print("✓ Share these with your mentor before running BindCraft.")

## Step 4 — Verify hotspot residues exist in the PDB

In [ ]:
if HOTSPOT_RESIDUES and TARGET_PDB_ID != "YOUR_PDB_ID":
    pdb_path = f"{WORK_DIR}/{TARGET_NAME}.pdb"
    if os.path.exists(pdb_path):
        with open(pdb_path) as f:
            lines = f.read().split("\n")
        
        # Get all residue numbers in the PDB
        pdb_residues = set()
        pdb_res_names = {}
        for line in lines:
            if line.startswith("ATOM") and len(line) > 26:
                resnum = int(line[22:26].strip())
                resname = line[17:20].strip()
                pdb_residues.add(resnum)
                pdb_res_names[resnum] = resname
        
        print("Hotspot residue verification:")
        all_ok = True
        for res in HOTSPOT_RESIDUES:
            if res in pdb_residues:
                aa3 = pdb_res_names[res]
                print(f"  ✓ Residue {res} ({aa3}) — found in PDB")
            else:
                print(f"  ✗ Residue {res} — NOT FOUND in PDB chain {TARGET_CHAIN}")
                print(f"    Check the residue numbering in Mol*!")
                all_ok = False
        
        print()
        if all_ok:
            print("✓ All hotspot residues verified. Ready to run BindCraft.")
        else:
            print("⚠ Some residues not found. Check PDB residue numbering in Mol* before proceeding.")
    else:
        print(f"PDB file not found at {pdb_path}. Run Step 2 first.")
else:
    print("Fill in TARGET_PDB_ID and HOTSPOT_RESIDUES in Steps 2 and 3 first.")

## Step 5 — Configure BindCraft settings

In [ ]:
if HOTSPOT_RESIDUES and TARGET_PDB_ID != "YOUR_PDB_ID":
    hotspot_str = ",".join(str(r) for r in HOTSPOT_RESIDUES)
    pdb_path = f"{WORK_DIR}/{TARGET_NAME}.pdb"
    
    settings = {
        "binder_name": TARGET_NAME,
        "starting_pdb": pdb_path,
        "chains": TARGET_CHAIN,
        "target_hotspot_residues": hotspot_str,
        "lengths": [65, 90],
        "number_of_final_designs": 10
    }
    
    settings_path = f"{WORK_DIR}/{TARGET_NAME}.json"
    with open(settings_path, "w") as f:
        json.dump(settings, f, indent=4)
    
    print("BindCraft Settings:")
    for k, v in settings.items():
        print(f"  {k}: {v}")
    print(f"\nSaved to: {settings_path}")
    print("\n✓ Ready to run BindCraft!")
else:
    print("Complete Steps 2 and 3 first.")

## Step 6 — Install BindCraft (if not already installed)

In [ ]:
%%bash
if [ ! -d "/content/bindcraft" ]; then
    git clone --depth 1 https://github.com/martinpacesa/BindCraft.git /content/bindcraft
    pip -q install "colabdesign @ git+https://github.com/sokrypton/ColabDesign"
    pip -q install pyrosetta-installer
    python -c "import pyrosetta_installer; pyrosetta_installer.install_pyrosetta()" 2>&1 | tail -2
    cd /content/bindcraft && bash install_bindcraft.sh --hardware=GPU 2>&1 | tail -3
    echo "✓ BindCraft installed"
else
    echo "✓ BindCraft already installed"
fi

## Step 7 — Run BindCraft on your target

**Expected time:** 1–3 hours depending on target and acceptance rate.

Watch the output. After 20 trajectories, check if any are being accepted. If 0 of 20 accept, call your mentor before continuing.

In [ ]:
if TARGET_PDB_ID == "YOUR_PDB_ID" or not HOTSPOT_RESIDUES:
    print("⚠ Complete Steps 2–5 before running BindCraft.")
else:
    import sys
    sys.path.insert(0, "/content/bindcraft")
    
    import pyrosetta
    pyrosetta.init("-mute all")
    
    from bindcraft import BindCraft
    
    bc = BindCraft(
        settings_path=settings_path,
        design_path=WORK_DIR,
        advanced_settings={
            "design_protocol": "default",
            "prediction_protocol": "default",
            "interface_protocol": "default",
            "template_protocol": "default"
        },
        filters={
            "i_pTM": 0.55, "pLDDT": 0.85, "Binder_pLDDT": 0.80,
            "i_pAE": 10, "ShapeComplementarity": 0.55,
            "dG": -10, "Clashes": 5, "Binder_RMSD": 2.0
        }
    )
    
    bc.run()
    
    print("\n" + "="*50)
    print(f"BindCraft run complete for {TARGET_NAME}!")
    print(f"Results saved to: {WORK_DIR}/{TARGET_NAME}/")
    print("="*50)

## Step 8 — Preliminary analysis of your results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f"{WORK_DIR}/{TARGET_NAME}/final_design_stats.csv"
failure_csv = f"{WORK_DIR}/{TARGET_NAME}/failure_csv.csv"

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv).sort_values("Average_i_pTM", ascending=False)
    
    n_accepted = len(df)
    n_failed = len(pd.read_csv(failure_csv)) if os.path.exists(failure_csv) else "unknown"
    
    print(f"=== {TARGET_NAME} BindCraft Results ===")
    print(f"  Accepted designs: {n_accepted}")
    print(f"  Rejected designs: {n_failed}")
    print()
    
    key_cols = [c for c in [
        "binder_name", "Average_i_pTM", "Average_ShapeComplementarity",
        "Average_Binder_RMSD", "Average_dG"
    ] if c in df.columns]
    
    print(df[key_cols].to_string())
    
    # Quick plot
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(df["Average_i_pTM"], bins=min(10, n_accepted), color="#B76E79", edgecolor="white")
    ax.axvline(0.55, color="red", linestyle="--", label="threshold")
    ax.set_title(f"{TARGET_NAME}: i_pTM Distribution ({n_accepted} designs)")
    ax.set_xlabel("Average i_pTM")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f"Results CSV not found at {results_csv}")
    print("BindCraft may still be running, or the run hasn't started yet (Step 7).")

## Step 9 — Compare to PD-L1 results

In [ ]:
pdl1_csv = "/content/drive/MyDrive/bindcraft/PDL1_8aok/final_design_stats.csv"
target_csv = f"{WORK_DIR}/{TARGET_NAME}/final_design_stats.csv"

if os.path.exists(pdl1_csv) and os.path.exists(target_csv):
    pdl1 = pd.read_csv(pdl1_csv)
    target = pd.read_csv(target_csv)
    
    compare_metrics = [
        "Average_i_pTM", "Average_ShapeComplementarity",
        "Average_Binder_RMSD", "Average_dG"
    ]
    compare_metrics = [m for m in compare_metrics if m in pdl1.columns and m in target.columns]
    
    print("=== PD-L1 vs. Your Target: Comparison ===")
    print(f"{'Metric':35s} {'PD-L1 mean':>12} {'Your target':>12}")
    print("-" * 61)
    for m in compare_metrics:
        print(f"{m:35s} {pdl1[m].mean():>12.3f} {target[m].mean():>12.3f}")
    print()
    print(f"{'N accepted':35s} {len(pdl1):>12} {len(target):>12}")
    print()
    print("Interpretation:")
    if len(target) >= len(pdl1) * 0.8:
        print("  Similar acceptance rate → your target is roughly as designable as PD-L1.")
    elif len(target) >= len(pdl1) * 0.3:
        print("  Lower acceptance rate → your target surface is harder to design against.")
    else:
        print("  Much lower acceptance rate → consider trying different hotspot residues.")
else:
    print("Need both datasets to compare.")
    if not os.path.exists(pdl1_csv):
        print("  PD-L1 dataset (PDL1_8aok) not found on Drive.")
    if not os.path.exists(target_csv):
        print("  Your target results not found — complete Step 7 first.")

---
## Next steps

Once you have results:
1. Run the **NB05 CSV analysis** on your target's `final_design_stats.csv`
2. Run **NB06 visualization** on your target's `Accepted/Ranked/` PDB files
3. Run **NB07 failure analysis** on your target's `failure_csv.csv`
4. Select your top 5 using **NB08**
5. Prepare your **final presentation** (see NB10 and wiki Module 9.6)

**Next:** Notebook 10 — Final analysis and presentation preparation